In [1]:
# =========================================================
# Notebook 02 — Academy Mapper / Entity Resolution
# =========================================================
#
# Purpose:
# Construct a URN -> URN_FINAL mapping that links predecessor
# and successor schools across institutional change, allowing
# longitudinal records to be stitched into consistent school
# entities for downstream modelling.
#
# Inputs:
# - gias_links.csv
#   Stored in:
#   - data/raw/
#
# Outputs:
# - urn_mapping.pkl
#   Stored in:
#   - data/processed/
#
# Role in the pipeline:
# This notebook performs entity resolution. It is a critical
# step because academy conversion, merger, and structural
# change can fragment school histories across multiple URNs.
# The resulting mapping is used in later notebooks to create
# a longitudinal institutional panel.
#
# Reproducibility note:
# The file paths in this notebook are currently configured for
# the author’s local machine. These should be updated if the
# project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
#
# Methodological note:
# This mapping uses directional link information from GIAS
# link records to follow predecessor/successor chains and
# resolve each school to its final institutional endpoint.
#
# Split handling note:
# One-to-many split cases are self-mapped rather than dropped.
# This design choice prevents downstream merge failures and
# preserves auditability. Such cases can still be filtered
# later if stricter continuity rules are required.
# =========================================================

import pandas as pd
import networkx as nx
import os
import pickle

# =========================================================
# Configuration
# =========================================================
#
# This section defines the local input and output paths used
# for entity resolution.
#
# - LINKS_FILE_PATH points to the raw GIAS links file.
# - OUTPUT_FILE is the serialized mapping artefact produced
#   by this notebook.
#
# If reproducing this pipeline on another machine, update
# these paths before running the notebook.
# =========================================================

# --- CONFIGURATION ---
BASE_DIR = r"C:\Users\kiero\Documents\msc-dissertation\data"
RAW_FOLDER = os.path.join(BASE_DIR, "raw")
PROCESSED_FOLDER = os.path.join(BASE_DIR, "processed")

LINKS_FILE_PATH = os.path.join(RAW_FOLDER, "gias_links.csv")
OUTPUT_FILE = os.path.join(PROCESSED_FOLDER, "urn_mapping.pkl")

print("🚀 Starting Work Package 2: Academy Mapper (Local Mode)\n")

# =========================================================
# Step 1 — Load Links Data
# =========================================================
#
# This step loads the GIAS links file containing relationships
# between schools.
#
# Required fields:
# - URN
# - LINKURN
# - LINKTYPE
#
# These fields are used to infer predecessor/successor
# directionality and build the institutional linkage graph.
# =========================================================

# ---------------------------------------------------------
# STEP 1: LOAD LINKS DATA
# ---------------------------------------------------------
print(f"1. Loading links data from: {LINKS_FILE_PATH}...")

if not os.path.exists(LINKS_FILE_PATH):
    raise FileNotFoundError(f"❌ Could not find {LINKS_FILE_PATH}. Please check the filename/path.")

try:
    df_links = pd.read_csv(LINKS_FILE_PATH, encoding="utf-8-sig", low_memory=False)
except UnicodeDecodeError:
    df_links = pd.read_csv(LINKS_FILE_PATH, encoding="latin1", low_memory=False)

print(f"✅ File loaded successfully! Rows: {len(df_links):,}")

# Normalize columns
df_links.columns = [c.strip().upper() for c in df_links.columns]

required = {"URN", "LINKURN", "LINKTYPE"}
missing = required - set(df_links.columns)
if missing:
    print("⚠️ Columns found:", df_links.columns.tolist())
    raise ValueError(f"Missing required columns: {missing}")

# Clean + types
df_links = df_links.dropna(subset=["URN", "LINKURN"]).copy()
df_links["URN"] = pd.to_numeric(df_links["URN"], errors="coerce")
df_links["LINKURN"] = pd.to_numeric(df_links["LINKURN"], errors="coerce")
df_links = df_links.dropna(subset=["URN", "LINKURN"])
df_links["URN"] = df_links["URN"].astype(int)
df_links["LINKURN"] = df_links["LINKURN"].astype(int)
df_links["LINKTYPE"] = df_links["LINKTYPE"].astype(str)

# Optional: parse dates if present (not required for mapping)
if "LINKESTABLISHEDDATE" in df_links.columns:
    df_links["LINKESTABLISHEDDATE"] = pd.to_datetime(
        df_links["LINKESTABLISHEDDATE"], errors="coerce", dayfirst=True
    )

# =========================================================
# Step 2 — Build Directional Graph
# =========================================================
#
# This step converts the tabular GIAS link data into a
# directed graph.
#
# Edge orientation is inferred from LINKTYPE:
# - "Predecessor ..." means LINKURN -> URN
# - "Successor ..."   means URN -> LINKURN
#
# The resulting directed graph allows institutional chains
# to be followed through time until a final sink node is
# reached.
# =========================================================

# ---------------------------------------------------------
# STEP 2: BUILD DIRECTIONAL GRAPH USING LINKTYPE
# ---------------------------------------------------------
print("\n2. Building Academy Network Graph (directional)...")

G = nx.DiGraph()

def orient_edge(row):
    """
    Your LINKS schema:
      URN = the 'main' school
      LINKURN = linked school
      LINKTYPE determines direction:

    - 'Predecessor ...' => LINKURN is predecessor of URN => LINKURN -> URN
    - 'Successor ...'   => LINKURN is successor of URN   => URN -> LINKURN
    """
    lt = row["LINKTYPE"].lower()
    u = row["URN"]
    v = row["LINKURN"]

    if "predecessor" in lt:
        return (v, u)
    if "successor" in lt:
        return (u, v)
    return None  # ignore other/unknown link types

edges = df_links.apply(orient_edge, axis=1).dropna().tolist()
G.add_edges_from(edges)

print(f"   - Nodes (Schools): {G.number_of_nodes():,}")
print(f"   - Edges (Links):   {G.number_of_edges():,}")

# =========================================================
# Step 3 — Resolve Institutional Chains
# =========================================================
#
# This step follows successor links until a terminal school
# identity is reached.
#
# Resolution rules:
# - If a school has no successor, it maps to itself.
# - If a school resolves cleanly through a one-to-one chain,
#   it maps to the final sink node.
# - If a cycle is detected, the node is self-mapped as a
#   safe fallback.
# - If a one-to-many split is encountered, the node is also
#   self-mapped to preserve downstream merge integrity.
#
# Memoization is used to improve performance on large graph
# traversals.
# =========================================================

# ---------------------------------------------------------
# STEP 3: RESOLVE CHAINS FAST (CACHED)
# ---------------------------------------------------------
print("\n3. Resolving broken time series (cached)...")

# Precompute successors for speed
succ_map = {n: list(G.successors(n)) for n in G.nodes()}

resolved_cache = {}  # node -> final sink
final_mapping = {}   # node -> final sink
excluded_splits = 0
cycle_hits = 0

def resolve_final(u: int):
    """
    Follow successor links until sink.

    Returns:
      - final URN (int) if resolved to a single sink
      - None if split (one-to-many successors encountered)

    Cycle handling:
      - if cycle detected, self-map (u) and count it

    Uses memoization to avoid repeated traversal.
    """
    global cycle_hits  # notebook-safe

    if u in resolved_cache:
        return resolved_cache[u]

    visited = set()
    path = []
    current = u

    while True:
        if current in resolved_cache:
            final = resolved_cache[current]
            break

        if current in visited:
            cycle_hits += 1
            final = u  # safest fallback
            break

        visited.add(current)
        path.append(current)

        succ = succ_map.get(current, [])
        if len(succ) == 0:
            final = current
            break
        if len(succ) > 1:
            final = None  # split (one-to-many)
            break

        current = succ[0]

    # Memoize resolved path if not split
    if final is not None:
        for n in path:
            resolved_cache[n] = final

    return final

nodes = list(G.nodes())
n_total = len(nodes)

for i, node in enumerate(nodes, start=1):
    node = int(node)
    final = resolve_final(node)

    if final is None:
        # ✅ CHANGE IMPLEMENTED: SELF-MAP split nodes so downstream merges never fail
        excluded_splits += 1
        final_mapping[node] = node
        continue

    final_mapping[node] = int(final)

    # progress ping
    if i % 5000 == 0:
        print(f"   ...resolved {i:,}/{n_total:,}")

print("\n✅ Mapping Complete.")
print(f"   - Schools Mapped (incl. self-mapped splits): {len(final_mapping):,}")
print(f"   - Split cases self-mapped (1→many):         {excluded_splits:,}")
print(f"   - Cycle guards triggered:                   {cycle_hits:,}")

# Quick sanity: how many changed?
changed = sum(1 for k, v in final_mapping.items() if k != v)
print(f"   - URNs that map to a new final URN:         {changed:,}")
print("   - Example changes:", [(k, v) for k, v in final_mapping.items() if k != v][:10])

# =========================================================
# Step 4 — Save Mapping Artefact
# =========================================================
#
# The resolved URN mapping is serialized as a pickle object
# for use in downstream panel construction and feature
# engineering notebooks.
# =========================================================


# ---------------------------------------------------------
# STEP 4: SAVE ARTEFACT
# ---------------------------------------------------------
print(f"\n4. Saving Mapping Artefact to: {OUTPUT_FILE}")
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(final_mapping, f)

print("🎉 DONE. You can now use this mapping to merge historical records.")

# =========================================================
# Outputs Produced
# =========================================================
#
# After successful execution, this notebook produces:
#
# - urn_mapping.pkl
#
# This mapping is used to reconcile predecessor/successor
# school identities and create longitudinal institutional
# records in later stages of the modelling pipeline.
#
# Next notebook in pipeline:
# - 03_feature_engineering.ipynb
# =========================================================

🚀 Starting Work Package 2: Academy Mapper (Local Mode)

1. Loading links data from: C:\Users\kiero\Documents\msc-dissertation\data\raw\gias_links.csv...
✅ File loaded successfully! Rows: 34,828

2. Building Academy Network Graph (directional)...
   - Nodes (Schools): 30,285
   - Edges (Links):   17,485

3. Resolving broken time series (cached)...
   ...resolved 5,000/30,285
   ...resolved 10,000/30,285
   ...resolved 15,000/30,285
   ...resolved 20,000/30,285
   ...resolved 25,000/30,285
   ...resolved 30,000/30,285

✅ Mapping Complete.
   - Schools Mapped (incl. self-mapped splits): 30,285
   - Split cases self-mapped (1→many):         146
   - Cycle guards triggered:                   6
   - URNs that map to a new final URN:         17,206
   - Example changes: [(134643, 100006), (100012, 100021), (100016, 132245), (100017, 132245), (100024, 100023), (100038, 100037), (100068, 100071), (100083, 100074), (100093, 100096), (126575, 100096)]

4. Saving Mapping Artefact to: C:\Users\kier